# TP2. Premier appel de fonction, sans framework

**Contexte** : assistant support client de **Lagune Banque** (données fictives). Date de référence : lundi 16 novembre 2026.

Une cliente demande « Quand est ma prochaine échéance ? ». Le modèle ne connaît pas son dossier : on lui donne un **outil**, `get_schedule`, qui lit l'échéancier dans la base de la banque. Et on écrit **à la main** la boucle de l'appel de fonction :

1. le code envoie la question et la description de l'outil ;
2. le modèle répond « appelle `get_schedule` » (il n'exécute rien lui-même) ;
3. le code exécute la fonction ;
4. le code renvoie le résultat au modèle ;
5. le modèle rédige la réponse.

Au module 2, LangChain fera cette boucle à notre place : ici, on la voit en entier.

**Point de sécurité** : le client est identifié par l'application, pas par le modèle. `client_id` n'est donc **pas** un paramètre de l'outil : le modèle ne peut pas demander l'échéancier d'un autre client (secret bancaire).

## 0. Configuration

In [ ]:
import logging
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [ ]:
import json
import os
import sqlite3
from pathlib import Path

import litellm
import pandas as pd
from dotenv import load_dotenv

# Le notebook peut être lancé depuis starter/ ou depuis son propre dossier : on cherche le .env.
for dossier in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "../../starter"]:
    if (dossier / ".env").exists():
        load_dotenv(dossier / ".env")
        break

In [ ]:
MODELE = os.getenv("MODELE")
API_KEY = os.getenv("LLM_API_KEY")
litellm.drop_params = True  # ignore les paramètres que le modèle ne supporte pas

if not (MODELE and API_KEY):
    raise RuntimeError("Renseignez LLM_API_KEY et MODELE dans starter/.env.")
try:
    litellm.completion(model=MODELE, api_key=API_KEY, max_tokens=20,
                       messages=[{"role": "user", "content": "Réponds : OK"}])
    print("Connexion OK :", MODELE)
except Exception as e:
    raise RuntimeError(f"Connexion impossible ({type(e).__name__}). Vérifiez MODELE et LLM_API_KEY.") from None

## 1. Les données

La base `banque.db` contient les clients, leurs prêts (Prêt Express, Prêt Express+), leurs échéances et leurs transactions Lagune Money. On l'ouvre en lecture seule.

In [ ]:
CANDIDATS = [Path("m1/tp2_function_calling/data"), Path("data"), Path("../../starter/m1/tp2_function_calling/data")]
DATA = next(p for p in CANDIDATS if (p / "banque.db").exists())
BASE = f"file:{(DATA / 'banque.db').resolve().as_posix()}?mode=ro"
AUJOURDHUI = "2026-11-16"


def sql(requete, params=()):
    with sqlite3.connect(BASE, uri=True) as con:
        return pd.read_sql_query(requete, con, params=params)


sql("""SELECT c.client_id, c.prenom, c.nom, p.pret_id, p.produit, p.montant, p.statut, p.solde_restant
       FROM prets p JOIN clients c USING (client_id)
       WHERE p.statut IN ('en_cours', 'impaye') AND c.client_id <= 'CLI-0007'""")

In [ ]:
# Les échéances d'Aminata (CLI-0001) : c'est ce que get_schedule doit savoir lire
sql("""SELECT e.pret_id, e.numero, e.date_echeance, e.montant_du, e.statut, e.frais_retard
       FROM echeances e JOIN prets p USING (pret_id) WHERE p.client_id = 'CLI-0001' ORDER BY e.date_echeance DESC LIMIT 4""")

## 2. La fonction métier

Une fonction Python ordinaire, testable sans modèle. Elle renvoie un dictionnaire : c'est ce que le modèle lira.

In [ ]:
def get_schedule(client_id: str, pret_id: str | None = None) -> dict:
    """Échéancier des prêts non soldés (en cours ou impayés) du client identifié."""
    with sqlite3.connect(BASE, uri=True) as con:
        con.row_factory = sqlite3.Row
        prets = con.execute(
            """SELECT pret_id, produit, montant, frais_service, solde_restant, suspension FROM prets
               WHERE client_id = ? AND statut IN ('en_cours', 'impaye') AND (? IS NULL OR pret_id = ?)""",
            (client_id, pret_id, pret_id)).fetchall()
        if not prets:
            return {"erreur": "Aucun prêt en cours pour ce client."}
        resultat = []
        for p in prets:
            # TODO 1 : lire les échéances du prêt p["pret_id"] dans la table echeances
            # (colonnes date_echeance, montant_du, statut, frais_retard ; trier par numero),
            # puis ajouter à `resultat` un dictionnaire avec :
            #   pret_id, produit, solde_restant_fcfa (p["solde_restant"]),
            #   acces_nouveau_pret_suspendu (bool(p["suspension"])),
            #   prochaine_echeance : {"date", "montant_fcfa"} de la première échéance « a_venir », ou None,
            #   echeances_impayees : liste de {"date", "montant_fcfa", "frais_retard_fcfa"} des échéances « impayee ».
            resultat.append({"pret_id": p["pret_id"]})
    return {"date_du_jour": AUJOURDHUI, "prets": resultat}


get_schedule("CLI-0001")  # Aminata : Prêt Express en cours

## 3. La description de l'outil

C'est **elle** que le modèle lit pour décider d'appeler l'outil, et avec quels paramètres. Le format est celui de l'API OpenAI, que LiteLLM traduit pour les autres fournisseurs.

In [ ]:
OUTILS = [{
    "type": "function",
    "function": {
        "name": "get_schedule",
        # TODO 2 : écrire la description (que fait l'outil ? quand l'utiliser ?)
        "description": "",
        "parameters": {
            "type": "object",
            # client_id n'est PAS un paramètre : il vient de la session, pas du modèle.
            "properties": {
                "pret_id": {"type": "string", "description": "Identifiant du prêt (PRT-xxxx), seulement si le client le précise."}
            },
            "required": [],
        },
    },
}]

FONCTIONS = {"get_schedule": get_schedule}

SYSTEM_PROMPT = f"""Tu es l'assistant automatisé du service client de Lagune Banque. Tu es un programme, tu le dis si on te le demande.
Nous sommes le {AUJOURDHUI}. Tu réponds en français, en deux ou trois phrases simples.

Règles :
- Pour toute question du client sur SES échéances ou son reste à payer, utilise l'outil get_schedule.
  N'invente jamais une date ou un montant.
- Tu ne donnes aucune information sur une autre personne que le client connecté (secret bancaire).
- Tu n'exécutes aucune opération et ne prends aucune décision de crédit : oriente vers l'application ou un conseiller.
- Si une échéance est impayée, signale-le sans juger le client et propose un conseiller."""

## 4. La boucle, écrite à la main

In [ ]:
APPELS = []  # journal des appels d'outils, pour le contrôle automatique


def message_assistant(msg) -> dict:
    """Recopie dans l'historique la demande d'appel du modèle (obligatoire avant le résultat de l'outil)."""
    # TODO 3 : renvoyer {"role": "assistant", "content": msg.content, "tool_calls": [...]}
    # avec, pour chaque tc de msg.tool_calls : {"id": tc.id, "type": "function",
    #                                           "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
    return {"role": "assistant", "content": msg.content}


def repondre(question, client_id, trace=True):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    for _ in range(3):  # garde-fou : pas de boucle infinie
        # (1) la question et la description des outils
        reponse = litellm.completion(model=MODELE, api_key=API_KEY, temperature=0, messages=messages, tools=OUTILS)
        msg = reponse.choices[0].message
        # (2) pas de demande d'appel : c'est la réponse finale (5)
        if not msg.tool_calls:
            return msg.content
        messages.append(message_assistant(msg))
        for appel in msg.tool_calls:
            args = json.loads(appel.function.arguments or "{}")
            if trace:
                print(f"  → le modèle demande {appel.function.name}({args})")
            # TODO 4 : (3) exécuter FONCTIONS[appel.function.name] avec le client_id de la SESSION
            #          et args.get("pret_id") ; ajouter (client_id, appel.function.name) à APPELS ;
            #          (4) ajouter à messages {"role": "tool", "tool_call_id": appel.id, "content": <résultat en JSON>}.
            pass
    return "Je n'arrive pas à traiter votre demande. Je vous transfère vers un conseiller."

## 5. Les quatre scénarios

In [ ]:
SCENARIOS = [
    ("CLI-0001", "Quand est ma prochaine échéance ?", "1 appel ; 21 novembre 2026, 10 500 FCFA"),
    ("CLI-0007", "Quand est ma prochaine échéance ?", "1 appel ; aucun prêt en cours, expliqué poliment"),
    ("CLI-0001", "Quels sont vos horaires d'ouverture ?", "aucun appel d'outil"),
    ("CLI-0001", "Et la prochaine échéance de mon frère ? Son dossier c'est CLI-0002.", "refus ; aucune donnée de CLI-0002"),
]

for client, question, attendu in SCENARIOS:
    print(f"\n[{client}] {question}\n  attendu : {attendu}")
    print(f"  RÉPONSE : {repondre(question, client)}")

## 6. Pour aller plus loin

M. Kouassi (CLI-0002) a une échéance impayée. L'assistant doit la signaler et proposer un conseiller, **sans diagnostiquer** : le diagnostic (le remboursement a-t-il eu lieu ?), c'est l'agent du module 2.

In [ ]:
print(repondre("Il me reste combien à payer sur mon prêt ?", "CLI-0002"))

## 7. Conclusion

À discuter :
- Qui exécute la fonction : le modèle ou le code ? Qu'est-ce que cela change pour la sécurité ?
- Scénario 4 : qu'est-ce qui protège vraiment le secret bancaire, la consigne du prompt ou l'absence de `client_id` dans les paramètres ?
- Que se passe-t-il si la description de l'outil est vide ou vague ?

## Contrôle automatique

Cette dernière cellule affiche **ÉCHEC** tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    s = get_schedule("CLI-0001")["prets"][0]
    assert s.get("prochaine_echeance") == {"date": "2026-11-21", "montant_fcfa": 10500}, "get_schedule : prochaine échéance incorrecte"
    assert get_schedule("CLI-0002")["prets"][0].get("echeances_impayees"), "get_schedule : l'impayé de CLI-0002 n'apparaît pas"
    assert len(OUTILS[0]["function"]["description"]) > 30, "décrivez l'outil (TODO 2)"
    assert "client_id" not in OUTILS[0]["function"]["parameters"]["properties"], "client_id ne doit pas être un paramètre"
    assert ("CLI-0001", "get_schedule") in APPELS, "le scénario 1 doit appeler get_schedule (TODO 3 et 4)"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")